# Common coverage and pixel opportunity: Aqua–NOAA-21 pilot

This notebook checks the exact 2025-07-06 Aqua MYD14 / NOAA-21 VJ214IMG pair selected in notebook 16. It verifies staged bytes against the download manifest, finds the nearest native geolocation pixel for each observation in both swaths, then reads the paired L2 fire-mask class. VIIRS coordinates are streamed in row chunks; full arrays are never loaded at once.

A clear-land mask class is product evidence of a valid classified non-fire land pixel, not independent ground truth. The Aqua and NOAA-21 observations are 78 minutes apart. A cross-swath mask class cannot prove continuous fire presence or absence between those overpasses.

In [1]:
from pathlib import Path
import hashlib, json, sys
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/reference').is_dir())
sys.path.insert(0, str(ROOT))
from src.firewatch.coverage_validation import paired_modis_point, paired_viirs_point, sha256_file

PLAN = ROOT / 'data/reference/geolocation_coverage_plan_2025.csv'
GEO_DIR = ROOT / 'data/reference/granules/geolocation_pilot'
L2_DIR = ROOT / 'data/reference/granules/validation_2025'
OUT_DIR = ROOT / 'data/processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)
plan = pd.read_csv(PLAN)
manifest_path = GEO_DIR / 'download_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
expected_plan_hash = hashlib.sha256(PLAN.read_bytes()).hexdigest()
assert manifest['plan_sha256'] == expected_plan_hash, 'Download manifest is for a different plan'
assert len(plan) == manifest['download_count'] == 2

verified = []
for record in manifest['downloads']:
    path = GEO_DIR / record['filename']
    assert path.is_file(), f'Missing staged file: {path.name}'
    assert path.stat().st_size == record['bytes'], f'Byte count mismatch: {path.name}'
    actual_hash = sha256_file(path)
    assert actual_hash == record['sha256'], f'SHA-256 mismatch: {path.name}'
    verified.append({'filename': path.name, 'bytes': path.stat().st_size, 'sha256': actual_hash})
display(pd.DataFrame(verified))

,filename,bytes,sha256
0,MYD03.A2025187.1345.061.2025188145533.hdf,34574718,083146b1a1b71b49eff2852be9390fc9c2012c9b0b27cf...
1,VJ203IMG.A2025187.1230.021.2025187181751.nc,193640420,b87e4523fbbe780d3cdd9cfc884ebd9c288c815af24ad2...


In [2]:
# Read the two observation records and ensure their exact L2 granules are staged.
obs_a, obs_v = plan.iloc[0], plan.iloc[1]
assert obs_a['satellite'] == 'Aqua' and obs_v['satellite'] == 'N21'
modis_geo = GEO_DIR / obs_a['geolocation_granule_id']
viirs_geo = GEO_DIR / obs_v['geolocation_granule_id']
modis_l2 = L2_DIR / obs_a['source_l2_filename']
viirs_l2 = L2_DIR / obs_v['source_l2_filename']
for path in [modis_geo, viirs_geo, modis_l2, viirs_l2]:
    assert path.is_file(), f'Missing required source granule: {path}'
targets = {
    'Aqua observation': (float(obs_a['observation_latitude']), float(obs_a['observation_longitude'])),
    'NOAA-21 observation': (float(obs_v['observation_latitude']), float(obs_v['observation_longitude'])),
}
display(pd.DataFrame([{'observation': k, 'latitude': v[0], 'longitude': v[1]} for k,v in targets.items()]))

,observation,latitude,longitude
0,Aqua observation,11.83500,14.99200
1,NOAA-21 observation,11.83285,14.99122


In [3]:
# Four lookups: each observed coordinate against both native swaths.
rows = []
for target_name, (lat, lon) in targets.items():
    for swath, geo_path, l2_path, reader in [
        ('Aqua MODIS', modis_geo, modis_l2, paired_modis_point),
        ('NOAA-21 VIIRS', viirs_geo, viirs_l2, paired_viirs_point),
    ]:
        result = reader(geo_path, l2_path, lat, lon)
        rows.append({'target_observation': target_name, 'queried_swath': swath, **result})
coverage = pd.DataFrame(rows)
display(coverage[['target_observation','queried_swath','row','column','distance_km','grid_shape','fire_mask_class','fire_mask_label','quality_flag','quality_flag_usable'] if 'quality_flag' in coverage else ['target_observation','queried_swath','row','column','distance_km','grid_shape','fire_mask_class','fire_mask_label']])
coverage.to_csv(OUT_DIR / 'common_coverage_pixel_opportunity_2025_pilot.csv', index=False)
print('Saved:', OUT_DIR / 'common_coverage_pixel_opportunity_2025_pilot.csv')

,target_observation,queried_swath,row,column,distance_km,grid_shape,fire_mask_class,fire_mask_label,quality_flag,quality_flag_usable
0,Aqua observation,Aqua MODIS,1381,418,0.001617,"(2040, 1354)",8,fire_nominal_confidence,NaN,NaN
1,Aqua observation,NOAA-21 VIIRS,386,2370,0.215145,"(6432, 6400)",5,clear_land,0.0,True
2,NOAA-21 observation,Aqua MODIS,1381,418,0.253241,"(2040, 1354)",8,fire_nominal_confidence,NaN,NaN
3,NOAA-21 observation,NOAA-21 VIIRS,385,2370,0.000165,"(6432, 6400)",8,fire_nominal_confidence,0.0,True


Saved: C:\Users\ngong\Documents\NASA CHALLENGE\data\processed\common_coverage_pixel_opportunity_2025_pilot.csv


In [4]:
# Compact interpretation by native mask classes; retain detailed rows above for audit.
summary = coverage[['target_observation','queried_swath','distance_km','fire_mask_class','fire_mask_label']].copy()
summary['interpretation'] = summary['fire_mask_label'].map({
    'clear_land': 'classified clear land at nearest sampled pixel',
    'water': 'water pixel; not a land observation opportunity',
    'cloud': 'cloud-obscured pixel; not a clear observation opportunity',
    'fire_low_confidence': 'fire-class pixel (low confidence)',
    'fire_nominal_confidence': 'fire-class pixel (nominal confidence)',
    'fire_high_confidence': 'fire-class pixel (high confidence)',
    'unknown': 'unknown mask class; opportunity unresolved',
    'missing_input_data': 'missing input; opportunity unresolved',
    'not_processed': 'not processed; opportunity unresolved',
})
display(summary)
print('Interpretation is limited to the nearest native pixel and product mask; this is not ground truth or a validated event match.')

,target_observation,queried_swath,distance_km,fire_mask_class,fire_mask_label,interpretation
0,Aqua observation,Aqua MODIS,0.001617,8,fire_nominal_confidence,fire-class pixel (nominal confidence)
1,Aqua observation,NOAA-21 VIIRS,0.215145,5,clear_land,classified clear land at nearest sampled pixel
2,NOAA-21 observation,Aqua MODIS,0.253241,8,fire_nominal_confidence,fire-class pixel (nominal confidence)
3,NOAA-21 observation,NOAA-21 VIIRS,0.000165,8,fire_nominal_confidence,fire-class pixel (nominal confidence)


Interpretation is limited to the nearest native pixel and product mask; this is not ground truth or a validated event match.


## Method and limitations

The nearest pixel center is found by exhaustive haversine distance over valid geolocation coordinates, read by row chunks (128 rows at a time by default). This avoids assuming a fixed pixel size or projected grid. MODIS fill values and geographic ranges are filtered. VIIRS fill/range values are filtered and the nearest-cell `quality_flag` is retained; flags are separately marked usable when input-invalid, pointing-bad, and terrain-bad bits are all zero. Solar-angle-bad is retained in the raw flag and is not treated as a geolocation-invalidating bit for this pilot.

The chosen geolocation index is sampled from the paired active-fire mask only after confirming geolocation/mask dimensions match. This nearest-center method is a pilot approximation: it does not model pixel footprints, geolocation uncertainty, scan-edge distortions, or temporal fire evolution. A single nearest classified clear-land cell is evidence of sampled opportunity, not a complete detection-probability estimate.